# Bevezetés a prompt tervezésbe
A prompt tervezés a természetes nyelvfeldolgozási feladatokhoz használt promptok megtervezésének és optimalizálásának folyamata. Ez magában foglalja a megfelelő promptok kiválasztását, a paramétereik hangolását és a teljesítményük értékelését. A prompt tervezés kulcsfontosságú a magas pontosság és hatékonyság eléréséhez a NLP modellekben. Ebben a szakaszban megvizsgáljuk a prompt tervezés alapjait az OpenAI modellek segítségével való felfedezés érdekében.


### 1. Gyakorlat: Tokenizáció
Fedezd fel a tokenizációt a tiktoken használatával, az OpenAI nyílt forráskódú, gyors tokenizálójával
További példákért nézd meg az [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) oldalt.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Gyakorlat 2: Microsoft Foundry Models kulcsbeállítás érvényesítése

> **Megjegyzés:** A GitHub Models 2026 júliusának végén leáll. Ez a gyakorlat helyette a [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) használatát javasolja, amely ugyanazt az ingyenesen kipróbálható modellikatalógust és az Azure AI Inference SDK élményt kínálja.

Futtasd le a lenti kódot, hogy ellenőrizd, helyesen van-e beállítva a Microsoft Foundry Models végpontod. A kód egy egyszerű, alapvető promptot próbál ki, és érvényesíti a választ. Az `oh say can you see` bemenetnek olyan választ kell adnia, mint a `by the dawn's early light..`


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

model_name = "gpt-4o-mini"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### Gyakorlat 3: Hamisítások
Vizsgáld meg, mi történik, ha arra kéred az LLM-et, hogy adjon választ egy olyan témával kapcsolatos felvetésre, ami talán nem létezik, vagy olyan témákra, amelyekről nem tudhat, mert kívül esnek a korábbi tanító adatállományán (frissebbek). Nézd meg, hogyan változik a válasz, ha másik felvetést vagy másik modellt próbálsz ki.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### 4. gyakorlat: Utasítás alapú
Használd a "text" változót a fő tartalom megadásához
és a "prompt" változót egy az adott fő tartalomhoz kapcsolódó utasítás megadásához.

Itt azt kérjük a modelltől, hogy összefoglalja a szöveget egy másodikos diák számára


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### 5. gyakorlat: Összetett kérés 
Próbálj ki egy olyan kérést, amely rendszer-, felhasználó- és asszisztensüzeneteket tartalmaz 
A rendszer beállítja az asszisztens kontextusát
A felhasználó és az asszisztens üzenetei többkörös beszélgetési kontextust biztosítanak

Figyeld meg, hogy az asszisztens személyisége "szarkasztikus"-ra van állítva a rendszer kontextusában. 
Próbálj ki egy másik személyiség-kontextust. Vagy próbálj ki egy másik be-/kimeneti üzenetsorozatot


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### Gyakorlat: Fedezd fel a megérzéseidet
A fenti példák mintákat adnak, amelyeket új feladatok létrehozására használhatsz (egyszerű, összetett, utasítás stb.) – próbálj meg más gyakorlatokat készíteni, hogy felfedezz néhány más ötletet, amikről beszéltünk, például példák, jelzések és még sok más.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
